# 同期实际EPS与预期版本：两个来源案例
保留原简报与同期报道的不同数值、样本口径及时间文字，不据近似或后来值配对。

In [1]:
from pathlib import Path
import os,json,hashlib
import pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/nvda_comparable_actual_cases_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/nvda_eps_corpus_20261015_v1';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us');parent,audit=_source(root,p['source'],['analysis/basis_comparison'])
if all((root/p['raw_dir']/e['file']).exists() for e in p['payloads']):raw={e['file']:(root/p['raw_dir']/e['file']).read_bytes() for e in p['payloads']}
else:
 old=json.loads((card/'summary.json').read_text())
 if not Path(old['recorder_local_dir']).exists():materialize_nas_evidence(card/'nas_binding.json',card=card)
 with R.uri_context(str(card/'qlib_recorder')):raw=R.get_recorder(recorder_id=old['recorder_id'],experiment_id=old['experiment_id']).load_object('inputs/raw_bytes')
for e in p['payloads']:assert hashlib.sha256(raw[e['file']]).hexdigest()==e['sha256']
receipt=json.loads(raw['receipt.json']);assert receipt['http_status']==200 and receipt['sha256']==hashlib.sha256(raw['morning_20250529.pdf']).hexdigest();text=raw['wire_evidence.txt'].decode()
for token in ['80 cents per share','estimates of 75 cents','96 cents','93 cents','17 analystes','15 avril']:assert token in text
original=parent['analysis/basis_comparison'].set_index('report_date');table=pd.DataFrame(p['cases'])
for row in table.itertuples():assert float(original.loc[row.report_date,'newsletter_consensus_eps'])==row.snapshot_consensus_eps
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/cases']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,raw_bytes=raw,parent_audit=audit);rec.save_objects(artifact_path='analysis',cases=table);rec.log_metrics(checked_cases=2,new_fits=0,new_backtests=0)
summary={'experiment':card.name,'decision':'needs_contemporaneous_comparable_actual_and_consensus_population','checked_cases':2,'qualified_training_cases':0,'snapshot_values_overwritten':False,'new_fits':0,'new_backtests':0,'objective_achieved':False,'provider_uri':str(provider),'region':'us','as_of':'2026-10-14','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope']};rec.save_objects(artifact_path='analysis',summary=summary);a=Path(summary['recorder_local_dir'])/'artifacts';rec.log_metrics(registered_artifact_count=sum(f.is_file() for f in a.rglob('*')));table.to_csv(card/'cases.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(table[['report_date','snapshot_consensus_eps','wire_expected_eps','wire_adjusted_actual_eps','wire_ex_charge_actual_eps']].to_string(index=False))


# 同期可比实际值：样本与版本不能拼接

固定核查两个前轮已识别的口径案例，未使用股价表现筛选事件或标签。本轮0拟合、0回测，不生成surprise，不改变已存预期或任何旧账户结果。

2019-02-14：已存Refinitiv DayAhead EPS预期0.75；Reuters署名、Business Standard转载的同期报道明确比较剔除项目后实际0.80与分析师预期0.75，与公司非GAAP数值一致。文章前段将营收预期归于IBES/Refinitiv，但EPS句没有单独标明供应商或样本范围，因此这是一项明确的新闻配对证据，仍不是完整I/B/E/S actual口径/历史版本认证。页面显示首次发布时间2019-02-15 03:15 IST、更新03:30 IST，保留原文，不将今天取得的网页自动当当年首次版本。

2025-05-28：原DayAhead普通预期及SmartEstimate均0.88；Reuters同期法文转载同时给出普通调整后实际0.81、剔H20费用及税项后的0.96，以及LSEG预期0.93。0.93段明确提及4月15日限制披露后提供预测的17位分析师；旧0.88快照没有相同样本范围证据。数值差可能涉及时间、样本或定义，不能只称“修订了5美分”，也不能把0.93覆盖原0.88。

该报道没有清晰消除“究竟哪个actual与哪组预期完全同口径”的问题。不能因为0.96与次年简报去年值相同就认证，也不能选择会产生正surprise的组合。0.81、0.96、0.88、0.93四个事实保留为不同字段，不拼成训练标签；下一年旧值禁止回填上年。

正常HTTPS下载LSEG 2025-05-29 Morning News Call成功，原PDF已冻结哈希及NAS归档，无TLS例外；第2页NVDA段落已视觉核验，仅讨论销售、指引和贸易限制，没有实际EPS/预期的数字配对。不能把“下载到同供应商晨报”当成口径缺口已解决。文件当前可得、页面/PDF日期均不独立证明历史首次公开时间。

来源：
- https://www.business-standard.com/amp/article/reuters/nvidia-beats-fourth-quarter-profit-estimates-shares-rise-119021500096_1.html
- https://www.boursorama.com/bourse/actualites/nvidia-prevoit-un-chiffre-d-affaires-inferieur-aux-estimations-les-restrictions-imposees-par-les-etats-unis-coutant-8-milliards-de-dollars-de-ventes-5685568571124616d4baa8eb40560a69
- https://share.lseg.com/assets/newsletters/Morning_News_Call/MNCGeneric_US_05292025.pdf

决定needs_contemporaneous_comparable_actual_and_consensus_population，合格训练案例仍为0。所需数据合同包含：当期可比actual的调整项目、对应预期的分析师样本/版本、生效与公开时点、提前预警。新闻出现同一数字只能补证，不能替代完整合同。该发现约束下一项预期差研究，尚不能解释旧策略亏损，也未实现稳定超额。生产保持不变。

原始工具文本、PDF和下载回执存Qlib Recorder/NAS；notebook核验哈希、关键数值文字与父Recorder两条原始预期一致，冷恢复逐对象哈希通过。没有新业务逻辑，不增加软件测试套件。保留两个案例的不同资格，不用一个报道配对成功外推全部35次事件。
